# Word Metadata Extracted from Frontiers in Neuroscience Submission: Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR^2 dataset](https://doi.org/10.71728/senscience.vsr2-qd06) using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library.

### Dataset Source
The dataset Croissant schema is at:

`https://sen.science/doi/10.71728/senscience.vsr2-qd06/fair2.json`

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading
Load Croissant dataset metadata and records using `mlcroissant`. The dataset metadata includes description, authors, version, and available record sets.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# The Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vsr2-qd06/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a structured object

# Print main metadata
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"Authors: {[a for a in getattr(metadata, 'author', [])]}")

## 2. Data Overview
The core metadata describes the available **record sets** (tables), their fields, and column IDs. 

We enumerate all `@id` values for record sets present in the dataset and, for each, list their fields.

In [ ]:
# List all record sets and fields with @id
print("Available Record Sets and their fields:")
record_sets = getattr(metadata, 'recordSet', [])
if not record_sets:
    print("No record sets defined in the metadata. (Data may be stored as a file-object based RecordSet.)")
    # Try to list via the internal mlcroissant helper
    # This dataset is structured around a single RecordSet
    try:
        internal_rs = dataset.list_record_sets()
        for rs in internal_rs:
            print(f"- RecordSet @id: {rs['@id']}, name: {rs.get('name', '')}")
            fields = rs.get('field', [])
            if not isinstance(fields, list):
                fields = [fields]
            print(f"  Fields/columns @id:")
            for f in fields:
                print(f"    - {f.get('@id', str(f))}")
    except Exception as e:
        print("Unable to enumerate record sets:", e)
else:
    for rs in record_sets:
        print(f"- RecordSet @id: {rs['@id']}, name: {rs.get('name', '')}")
        fields = rs.get('field', [])
        if not isinstance(fields, list):
            fields = [fields]
        print(f"  Fields/columns @id:")
        for f in fields:
            print(f"    - {f.get('@id', str(f))}")

# The alternative: enumerate data via dataset.records
print("\nExample records from main RecordSet:")
for i, x in enumerate(dataset.records(record_set='cr:RecordSet'), 1):
    print(x)
    if i >= 2:
        break

## 3. Data Extraction
Load the main record set as a DataFrame. All entity references (record set, fields) use their `@id`.

For this dataset, the main record set `@id` is `'cr:RecordSet'`.

In [ ]:
# Extract main record set as DataFrame
main_record_set_id = 'cr:RecordSet'
record_sets = [main_record_set_id]
dataframes = {}
for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

print(f'Columns in RecordSet {main_record_set_id}:')
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
We will analyze numeric and categorical fields in the `cr:RecordSet` table, applying basic filtering and transformations. All columns should be referenced by their `@id`.

**For this dataset, likely numeric fields are:**
- `cr:file_size` – file size in bytes

**Group/categorical fields may include:**
- `cr:file_extension` – document type (should be `.docx`)
- `cr:file_name` – file name

We demonstrate removing outliers and normalizing file sizes, then group by file extension.

In [ ]:
# Numeric field: File size in bytes
numeric_field_id = 'cr:file_size'
df = dataframes[main_record_set_id]

# If non-numeric, coerce
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

threshold = 30000  # Only show files >30 kB for illustration (these are big files for docx)
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Files with {numeric_field_id} > {threshold} bytes:")
print(filtered_df[['cr:file_name', numeric_field_id]].head())

# Normalization
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / (filtered_df[numeric_field_id].std() or 1)
print(f"\nNormalized {numeric_field_id}:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by file extension
group_field_id = 'cr:file_extension'
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"\nMean file size by extension:")
    print(grouped_df)

## 5. Visualization
Let's visualize the distribution of file sizes and a histogram grouped by extension.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

df = dataframes[main_record_set_id]
plt.figure(figsize=(7,4))
df['cr:file_size'] = pd.to_numeric(df['cr:file_size'], errors='coerce')

# Histogram of file sizes
plt.hist(df['cr:file_size'].dropna(), bins=10, color='skyblue', edgecolor='k')
plt.title('File Size Distribution (bytes)')
plt.xlabel('Size (bytes)')
plt.ylabel('Frequency')
plt.show()

# Bar chart by file extension
if 'cr:file_extension' in df.columns:
    group = df.groupby('cr:file_extension')['cr:file_size'].mean()
    group.plot(kind='bar', color='salmon', edgecolor='k', ylabel='Mean File Size (bytes)', title='Mean File Size by Extension')
    plt.show()

## 6. Conclusion
- The dataset contains metadata for Word documents submitted to Frontiers in Neuroscience (opinion and cover letter), including their file names, sizes, extensions, and checksums.
- File sizes are consistent with typical `.docx` scientific article and letter files.
- All Croissant entities and columns were accessed via their `@id`, ensuring robust and repeatable analyses.

For more details, refer to the [FAIR^2 Croissant package](https://doi.org/10.71728/senscience.vsr2-qd06) documentation.